# Pavan et al. (2014) Index-Policy Benchmark — `cont_single`

Estimates the **optimal expected revenue** for the bandit auction as defined in
`cont_single_colab.ipynb` (2 buyers, 1 good, continuous types, T=5, δ=0.9).

**Theory (PST 2014, Section 5, Proposition 2).** In the bandit auction, impulse
responses satisfy {i,t,s}=1$ for all ,t,s$. The virtual surplus simplifies to:
66783E[	ext{Rev}] = E\!\left[\sum_{t=0}^{T-1}\delta^t\sum_i
\left(	heta_{it} - rac{1-F_0(	heta_{i0})}{f_0(	heta_{i0})} - c_iight)x_{it}ight]66783

Key: the hazard rate /\eta_{i0}(	heta_{i0})$ is evaluated at the **initial type**
$	heta_{i0}$ and held **fixed** for the whole episode (IC rent from initial private info).

The **Gittins index** (Eq. 15) gives the optimal allocation:
66783\gamma_{it}(	heta_i^t) = \max_{	au \ge t}
rac{E\left[\sum_{s=t}^{	au}\delta^{s-t}\left(	heta_{is}-c_i-
rac{1-F_0(	heta_{i0})}{f_0(	heta_{i0})}ight)igg|	heta_{it},
	ext{always allocated}ight]}{\sum_{s=t}^{	au}\delta^{s-t}}66783

Allocate to $\operatorname{argmax}_i \gamma_{it}$ if that max $\geq 0$.

We report two estimates:
- **Myerson (static)**: uses $\psi_i=v_i-(1-F_0(v_i))/f_0(v_i)$ at current type (lower bound).
- **Pavan (Gittins index)**: Eq. (15) with initial hazard rate (tight optimal benchmark).

In [ ]:
RUN_ENV = 'colab'    # 'colab' | 'local'
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Bandit Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
        'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU',
        'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime -> Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Bandit Auction'
    except Exception:
        PROJECT_DIR = os.getcwd()
else:
    PROJECT_DIR = LOCAL_PROJECT_DIR or os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## Imports & Auction Setup

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from bandit_auction.envs.cont_single import create_cont_single_auction
from bandit_auction.baselines.pavan import (
    eval_myerson_mechanism,
    eval_pavan_bandit_auction,
)

SEED = 42
print(f'JAX devices: {jax.devices()}')

In [ ]:
# Same auction configuration as cont_single_colab.ipynb
auction = create_cont_single_auction(
    n_buyers       = 2,
    n_goods        = 1,
    horizon        = 5,
    discount       = 0.9,
    type_mean      = 2.5,
    type_std       = 0.1,
    types_low      = 0.0,
    types_high     = 5.0,
    sigma_no       = 1e-2,
    trans_lambda   = 1.0,
    allocation_cost= 0.1,
    seed           = SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}, '
      f'discount={auction.discount}')
print(f'Type range: [{float(auction.types_low[0,0])}, {float(auction.types_high[0,0])}]  '
      f'sigma_no={auction.trans_sigma_no}')

## Myerson (Static) Baseline

Uses the per-period Myerson virtual value $\psi_i(v)=v-(1-F(v))/f(v)$,
ignoring the dynamic impulse-response terms.  
This is a **lower bound** on the optimal dynamic revenue.

In [ ]:
NUM_SAMPLES_MYERSON = 2000

print(f'Running Myerson baseline ({NUM_SAMPLES_MYERSON} trajectories)...')
myerson_rev, myerson_surplus = eval_myerson_mechanism(
    auction,
    num_samples = NUM_SAMPLES_MYERSON,
    seed        = SEED,
    mean        = 2.5,
    std         = 0.1,
)
print(f'  Revenue : {myerson_rev:.4f}')
print(f'  Surplus : {myerson_surplus:.4f}')

## Pavan (Gittins Index) Benchmark

Implements PST 2014 Eq. (15) correctly:
- $1/\eta_{i0}(\theta_{i0})$ computed **once at $t=0$** from the initial type, fixed thereafter.
- Gittins stopping-time index includes both $c_i$ and $1/\eta_{i0}(\theta_{i0})$ in the numerator.
- Revenue accumulated as $\delta^t(\theta_{it}-1/\eta_{i0}(\theta_{i0})-c_i)x_{it}$.

`num_index_samples` controls MC rollouts per Gittins index evaluation. 50 is a good balance.

In [ ]:
NUM_SAMPLES_PAVAN   = 500   # trajectories for revenue estimate
NUM_INDEX_SAMPLES   = 50    # MC rollouts per Gittins index evaluation

print(f'Running Pavan Gittins benchmark ({NUM_SAMPLES_PAVAN} trajectories, '
      f'{NUM_INDEX_SAMPLES} index samples)...')
pavan_rev, pavan_surplus = eval_pavan_bandit_auction(
    auction,
    num_samples       = NUM_SAMPLES_PAVAN,
    num_index_samples = NUM_INDEX_SAMPLES,
    seed              = SEED,
    mean              = 2.5,
    std               = 0.1,
)
print(f'  Revenue : {pavan_rev:.4f}')
print(f'  Surplus : {pavan_surplus:.4f}')

## Summary

In [ ]:
print('='*55)
print(f'  Benchmark revenue estimates (cont_single, seed={SEED})')
print('='*55)
print(f'  Myerson (static lower bound):  {myerson_rev:.4f}')
print(f'  Pavan   (dynamic, optimal  ):  {pavan_rev:.4f}')
print(f'  Dynamic correction          : +{pavan_rev - myerson_rev:.4f}')
print()
print('  Use PROFIT_GOAL in cont_single_colab.ipynb:')
print(f'    Tight goal  -> PROFIT_GOAL = {pavan_rev:.4f}')
print(f'    Conservative -> PROFIT_GOAL = {myerson_rev:.4f}')

## (Optional) Compare with Trained Mechanism

Load a trained mechanism from `cont_single_colab.ipynb` and compare its revenue
against the Pavan benchmark.

In [ ]:
import pickle
from bandit_auction.core import AuctionSimulator, BoundedContPolicy, build_bounded_policy_net
from bandit_auction.mechanisms.network import Mechanism, build_mechanism_network
from bandit_auction.train.evaluate import eval_mechanism_multikey, print_eval_report

EXPERIMENT = 'cont_single'
METHOD     = 'acceptance'   # 'acceptance' | 'penalty'
LOAD_SEED  = 42

load_dir = f'outputs/{EXPERIMENT}/{METHOD}/seed{LOAD_SEED}'

if not os.path.exists(f'{load_dir}/mechanism_params.pkl'):
    print(f'No saved params at {load_dir}/ — run cont_single_colab.ipynb first.')
else:
    with open(f'{load_dir}/mechanism_params.pkl', 'rb') as f:
        mechanism_params = pickle.load(f)
    with open(f'{load_dir}/policy_params.pkl', 'rb') as f:
        policy_params = pickle.load(f)

    mechanism_network = build_mechanism_network()
    policy_network = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=128, residual_init_std=1.0,
    )

    eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
    results = eval_mechanism_multikey(
        auction           = auction,
        mechanism_network = mechanism_network,
        mechanism_params  = mechanism_params,
        profit_goal       = pavan_rev,   # use Pavan revenue as the goal
        num_samples       = 256,
        keys              = eval_keys,
        policy_network    = policy_network,
        policy_params     = policy_params,
        is_discrete       = False,
    )
    print_eval_report(results, prefix=f'{EXPERIMENT} ({METHOD}): ')
    print(f'  Pavan benchmark revenue : {pavan_rev:.4f}')
    print(f'  Trained mechanism revenue: {results["profit_stats"]["mean"]:.4f}')
    gap = pavan_rev - results['profit_stats']['mean']
    print(f'  Gap to benchmark        : {gap:+.4f}')